# Robust Cross-Dataset Generalization of Machine Learning-Based Network Intrusion Detection Under Temporal and Distributional Shift

Исследовательский протокол. Run All запускает обучение, сравнение признаков, проверку калибровки и оценку на внешнем наборе UCI. Распакуйте весь проект, откройте notebook из его корневой папки. Для нового каталога результатов задайте NIDS_OUTPUT. Этапы исследования и источники указаны в README.txt.

In [1]:
from pathlib import Path
import os, json, sys, platform, hashlib, importlib.metadata
from threadpoolctl import threadpool_limits
threadpool_limits(limits=4)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
MODE=os.environ.get('NIDS_MODE','train')
assert MODE in {'replay','train'}
BASE=Path.cwd()
SAVED=Path(os.environ.get('NIDS_SAVED',str(BASE/'saved_results')))
FINAL=Path(os.environ.get('NIDS_OUTPUT',str(BASE/('run_results' if MODE=='train' else 'replay_results'))))
FINAL.mkdir(parents=True,exist_ok=True)
OUT=FINAL
SEED=42
BOOTSTRAPS=250
RUN_DATASETS=['UNSW_NB15','TON_IOT']
SAMPLE_RATES={'UNSW_NB15':.07,'TON_IOT':.015}
TIME_COLUMN={'UNSW_NB15':'meta_Stime','TON_IOT':'meta_ts'}
CAT_ALLOW={'UNSW_NB15':['proto','state','service'],'TON_IOT':['proto','conn_state','service']}
USE_RANDOM_FOREST=True
CORE=['log_duration','tcp','udp','icmp','other']
packages=['numpy','pandas','scipy','scikit-learn','lightgbm','pyarrow','matplotlib','nbformat','nbclient']
runtime={p:importlib.metadata.version(p) for p in packages}
(FINAL/'runtime.json').write_text(json.dumps(dict(mode=MODE,python=sys.version,platform=platform.platform(),packages=runtime),indent=2))
print('Mode:',MODE,'| output:',FINAL)


Mode: train | output: run_results


In [2]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier
from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon
import joblib
HAVE_LGBM=True
SCHEMAS={}
if MODE=='train':
    CLEAN_ROOT=Path(os.environ.get('NIDS_CLEAN_ROOT',str(BASE/'training_samples')))
    if not (CLEAN_ROOT/'UNSW_NB15'/'data').is_dir():
        candidates=sorted(Path('/kaggle/input').rglob('UNSW_NB15/manifest.json'))
        roots=[p.parent.parent for p in candidates if all((p.parent.parent/d/'schema.json').exists() and (p.parent.parent/d/'data').is_dir() for d in RUN_DATASETS+['CICIOT2023'])]
        assert len(roots)==1, 'Set NIDS_CLEAN_ROOT explicitly; need exactly one complete cleaned data root.'
        CLEAN_ROOT=roots[0]
    IS_SAMPLE=(CLEAN_ROOT/'sample_export.json').exists()
    if IS_SAMPLE:
        manifest=pd.read_csv(CLEAN_ROOT/'sample_sha256.csv')
        for row in manifest.itertuples(index=False):
            assert hashlib.sha256((CLEAN_ROOT/row.file).read_bytes()).hexdigest()==row.sha256, row.file
        print('Verified Kaggle sample hashes; sampling is not repeated.')
    for ds in RUN_DATASETS+['CICIOT2023']:
        assert all((CLEAN_ROOT/ds/q).exists() for q in ['manifest.json','schema.json','data']), f'Missing cleaned input: {ds}'
        assert list((CLEAN_ROOT/ds/'data').glob('*.parquet')), f'No Parquets: {ds}'
    forbidden={'target','label','attack_cat','attack_label','type','id','stime','ltime','ts','timestamp','srcip','dstip','src_ip','dst_ip'}
    for ds in RUN_DATASETS:
        schema=json.loads((CLEAN_ROOT/ds/'schema.json').read_text())
        nums=[c for c in schema['numeric_columns'] if c.lower() not in forbidden and not c.lower().startswith('meta_')]
        SCHEMAS[ds]={'num':nums,'cat':[c for c in CAT_ALLOW[ds] if c in schema['categorical_columns']],'time':TIME_COLUMN[ds]}
        assert nums and SCHEMAS[ds]['cat']
    inputs=[]
    for f in sorted(CLEAN_ROOT.rglob('*')):
        if f.is_file() and f.suffix in {'.json','.parquet'}:
            h=hashlib.sha256()
            with f.open('rb') as stream:
                for block in iter(lambda:stream.read(8*1024*1024),b''): h.update(block)
            inputs.append(dict(path=str(f.relative_to(CLEAN_ROOT)),bytes=f.stat().st_size,sha256=h.hexdigest()))
    pd.DataFrame(inputs).to_csv(FINAL/'training_input_manifest.csv',index=False)


Verified Kaggle sample hashes; sampling is not repeated.


In [3]:
if MODE=='train':
    def parse_event_time(series):
        text = series.astype('string').str.strip()
        val = pd.to_numeric(text, errors='coerce')
        secs = val.copy()
        ms = val.ge(1e12) & val.le(2.1e12)
        secs.loc[ms] = val.loc[ms] / 1000.0
        valid_num = secs.ge(946684800) & secs.le(2082758400)  # 2000–2035 UTC
        out = pd.Series(pd.NaT, index=series.index, dtype='datetime64[ns, UTC]')
        valid_num = valid_num.fillna(False)
        if valid_num.any():
            out.loc[valid_num] = pd.to_datetime(secs.loc[valid_num],unit='s',utc=True,errors='coerce')
        txt_mask = (~valid_num) & val.isna() & text.notna() & text.ne('')
        if txt_mask.any():
            out.loc[txt_mask] = pd.to_datetime(text.loc[txt_mask], utc=True, errors='coerce', format='mixed')
        out = out.where(out.between(pd.Timestamp('2000-01-01',tz='UTC'),
                                    pd.Timestamp('2035-12-31',tz='UTC')))
        return out

    def load_sample(ds):
        cfg = SCHEMAS[ds]
        cols = list(dict.fromkeys(cfg['num']+cfg['cat']+
                     [cfg['time'],'source_file','source_row','target','attack_label']))
        parts = sorted((CLEAN_ROOT/ds/'data').glob('*.parquet'))
        assert parts, f'No parquet data for {ds}'
        samples=[]
        selected=0
        for i,p in enumerate(parts,1):
            x=pd.read_parquet(p,columns=cols)
            h=pd.util.hash_pandas_object(x[['source_file','source_row']], index=False).to_numpy(dtype='uint64')
            u=(h & np.uint64(0xffffffff)).astype('float64') / 2**32
            x=x.copy() if IS_SAMPLE else x.loc[u<SAMPLE_RATES[ds]].copy()
            if len(x):
                x['_event_time']=parse_event_time(x[cfg['time']])
                x['target']=pd.to_numeric(x['target'],errors='raise').astype('int8')
                x['_row_key']=ds+'|'+x['source_file'].astype(str)+'|'+x['source_row'].astype(str)
                x=x.drop(columns=[cfg['time']])
                for col in cfg['num']:
                    x[col]=pd.to_numeric(x[col],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
                for col in cfg['cat']:
                    x[col]=x[col].astype('string').fillna('missing').replace({'':'missing','-':'missing'}).astype(str)
                samples.append(x)
                selected+=len(x)
            if i%60==0 or i==len(parts):
                print(f'{ds}: read {i}/{len(parts)} parquet parts; sampled {selected:,}',flush=True)
        df=pd.concat(samples,ignore_index=True)
        predictors=cfg['num']+cfg['cat']
        df['_group']=pd.util.hash_pandas_object(df[predictors], index=False).to_numpy(dtype='uint64')
        assert df['_row_key'].is_unique, f'{ds}: repeated original row identities'
        assert df['target'].isin([0,1]).all()
        return df

    DATA={ds:load_sample(ds) for ds in RUN_DATASETS}
    rows=[]; audit_classes=[]
    for ds,x in DATA.items():
        valid=x['_event_time'].notna()
        t=x.loc[valid,'_event_time']
        rows.append({'dataset':ds,'sampled':len(x),'valid_timestamps':int(valid.sum()),
                     'valid_time_pct':round(100*valid.mean(),3),
                     'first_utc':str(t.min()) if len(t) else '',
                     'last_utc':str(t.max()) if len(t) else '',
                     'distinct_timestamps':t.nunique(),
                     'distinct_dates':t.dt.floor('D').nunique(),
                     'largest_timestamp_fraction':round(t.value_counts(normalize=True).max(),4) if len(t) else np.nan,
                     'global_attack_pct':round(100*x.target.mean(),3),
                     'unique_feature_groups':int(x._group.nunique())})
        if len(t):
            z=x.loc[valid,['target','attack_label','_event_time']].copy()
            z['utc_day']=z['_event_time'].dt.strftime('%Y-%m-%d')
            g=z.groupby('utc_day').agg(n=('target','size'),attacks=('target','sum')).reset_index()
            g['attack_pct']=100*g['attacks']/g['n']
            g['dataset']=ds
            audit_classes.append(g)
    AUDIT=pd.DataFrame(rows)
    DAY=pd.concat(audit_classes,ignore_index=True) if audit_classes else pd.DataFrame()
    AUDIT.to_csv(OUT/'temporal_integrity_audit.csv',index=False)
    DAY.to_csv(OUT/'temporal_daily_class_balance.csv',index=False)
    display(AUDIT)
    for ds in RUN_DATASETS:
        print('\n',ds,'daily class balance (first/last):')
        display(DAY[DAY.dataset==ds].head(4))
        display(DAY[DAY.dataset==ds].tail(4))


UNSW_NB15: read 1/1 parquet parts; sampled 177,906


TON_IOT: read 1/1 parquet parts; sampled 179,675


,dataset,sampled,valid_timestamps,valid_time_pct,first_utc,last_utc,distinct_timestamps,distinct_dates,largest_timestamp_fraction,global_attack_pct,unique_feature_groups
0,UNSW_NB15,177906,177906,100.0,2015-01-22 11:49:37+00:00,2015-02-18 12:21:08+00:00,71662,3,0.0001,12.587,159103
1,TON_IOT,179675,179675,100.0,2019-04-02 09:49:44+00:00,2019-04-27 01:25:17+00:00,67095,8,0.0005,96.562,173735



 UNSW_NB15 daily class balance (first/last):


,utc_day,n,attacks,attack_pct,dataset
0,2015-01-22,73987,1532,2.070634,UNSW_NB15
1,2015-01-23,2335,0,0.000000,UNSW_NB15
2,2015-02-18,101584,20861,20.535714,UNSW_NB15


,utc_day,n,attacks,attack_pct,dataset
0,2015-01-22,73987,1532,2.070634,UNSW_NB15
1,2015-01-23,2335,0,0.000000,UNSW_NB15
2,2015-02-18,101584,20861,20.535714,UNSW_NB15



 TON_IOT daily class balance (first/last):


,utc_day,n,attacks,attack_pct,dataset
3,2019-04-02,654,0,0.000000,TON_IOT
4,2019-04-03,1839,0,0.000000,TON_IOT
5,2019-04-04,527,0,0.000000,TON_IOT
6,2019-04-23,26846,26637,99.221486,TON_IOT


,utc_day,n,attacks,attack_pct,dataset
7,2019-04-24,9618,9353,97.244749,TON_IOT
8,2019-04-25,78362,77139,98.439295,TON_IOT
9,2019-04-26,58722,57368,97.694220,TON_IOT
10,2019-04-27,3107,3000,96.556164,TON_IOT


In [4]:
def purge_groups(x):
    order = {'train': 0, 'calibration': 1, 'test': 2}
    y = x.sort_values(['_event_time', '_row_key']).copy() if '_event_time' in x else x.copy()
    y['_priority'] = y['split'].map(order)
    first = y.sort_values(['_priority', '_event_time', '_row_key']).drop_duplicates('_group', keep='first')
    chosen_period = first.set_index('_group')['split']
    mask = y['split'].eq(y['_group'].map(chosen_period))
    dropped = y.loc[~mask].groupby('split').size().to_dict()
    result = y.loc[mask].drop(columns=['_priority']).copy()
    result.attrs['purged_cross_period'] = dropped
    return result


In [5]:
def signed_log1p(x):
    return np.sign(x)*np.log1p(np.abs(x))

try:
    OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    def onehot(): return OneHotEncoder(handle_unknown='ignore',sparse_output=False)
except TypeError:
    def onehot(): return OneHotEncoder(handle_unknown='ignore',sparse=False)

def build_preprocessor(ds):
    cfg=SCHEMAS[ds]
    numeric=Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),
                      ('log',FunctionTransformer(signed_log1p,validate=False)),
                      ('scale',StandardScaler())])
    cat=Pipeline([('onehot',onehot())])
    return ColumnTransformer([('num',numeric,cfg['num']),('cat',cat,cfg['cat'])],sparse_threshold=0)

def models(ds):
    pool={
        'LogisticRegression':LogisticRegression(max_iter=350,random_state=SEED),
    }
    if HAVE_LGBM:
        pool['LightGBM']=LGBMClassifier(n_estimators=220,learning_rate=.05,num_leaves=15,
           max_depth=6,min_child_samples=80,reg_lambda=8.0,verbosity=-1,random_state=SEED,n_jobs=4)
    if USE_RANDOM_FOREST:
        pool['RandomForest']=RandomForestClassifier(n_estimators=110,max_depth=16,
            min_samples_leaf=5,n_jobs=-1,random_state=SEED)
    return {name:Pipeline([('preprocess',build_preprocessor(ds)),('model',model)]) for name,model in pool.items()}

def fpr_threshold(cal_y,cal_prob,alpha=.01):
    neg=np.asarray(cal_prob)[np.asarray(cal_y)==0]
    assert len(neg)>=30
    # Threshold strictly above empirical (1-alpha) percentile of benign scores.
    vals=np.sort(neg)
    k=max(0,min(len(vals)-1,int(np.ceil((1-alpha)*len(vals)))-1))
    return float(np.nextafter(vals[k],np.inf))

def wilson(k,n,z=1.9599639845):
    if n==0:return np.nan,np.nan
    p=k/n; den=1+z*z/n
    mid=(p+z*z/(2*n))/den
    hw=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return max(0,mid-hw),min(1,mid+hw)

def f1_fixed_prior(tpr,fpr,prior=.2):
    tp=prior*tpr;fp=(1-prior)*fpr;fn=prior*(1-tpr)
    return 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn>0 else 0.

def boot_f1_interval(y,pred,seed=SEED,reps=BOOTSTRAPS):
    rng=np.random.default_rng(seed)
    y=np.asarray(y);p=np.asarray(pred)
    pp=p[y==1]; pn=p[y==0]
    pos_n=len(pp);neg_n=len(pn)
    if not pos_n or not neg_n:return np.nan,np.nan
    vals=[]
    for _ in range(reps):
        tp=int(rng.choice(pp,size=pos_n,replace=True).sum())
        fp=int(rng.choice(pn,size=neg_n,replace=True).sum())
        fn=pos_n-tp
        vals.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.)
    return np.quantile(vals,[.025,.975]).tolist()

def evaluate(y,prob,threshold):
    y=np.asarray(y,dtype='int8');prob=np.asarray(prob,dtype='float64')
    pred=(prob>=threshold).astype('int8')
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    rec=tp/(tp+fn) if tp+fn else np.nan
    fpr=fp/(tn+fp) if tn+fp else np.nan
    prec=tp/(tp+fp) if tp+fp else 0
    f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
    f1_ben=2*tn/(2*tn+fp+fn) if 2*tn+fp+fn else 0
    f1lo,f1hi=np.nan,np.nan  # use cluster intervals below; avoid iid row bootstrap
    rlo,rhi=wilson(tp,tp+fn);flo,fhi=wilson(fp,fp+tn)
    return {'n':len(y),'attack_pct':100*y.mean(),
            'accuracy':(tp+tn)/len(y),'precision':prec,'recall':rec,'f1':f1,
            'macro_f1':(f1+f1_ben)/2,'f1_at_20pct_attack':f1_fixed_prior(rec,fpr),
            'roc_auc':roc_auc_score(y,prob) if len(set(y))==2 else np.nan,
            'average_precision':average_precision_score(y,prob) if len(set(y))==2 else np.nan,
            'fpr':fpr,'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),
            'f1_ci_low':f1lo,'f1_ci_high':f1hi,
            'recall_ci_low':rlo,'recall_ci_high':rhi,
            'fpr_ci_low':flo,'fpr_ci_high':fhi}


In [6]:
if MODE=='train':
    # Strict calendar-day validation; reuses the existing label-independent samples.
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import json, joblib, platform, sys
    from sklearn.base import clone

    import sys
    sys.path.insert(0,str(BASE/'source'))
    from external_eval import read_release, source_overlap, score_external, evaluate_external, calibration_analysis
    EXT_X, EXT_Y = read_release(BASE)
    EXT_RUNS = []
    CAL_ROWS = []
    STRICT=[]; STRICT_PRED=[]; STRICT_SPLITS=[]
    for ds,raw in DATA.items():
        x=raw.dropna(subset=['_event_time']).copy()
        day=x._event_time.dt.strftime('%Y-%m-%d')
        if ds=='UNSW_NB15':
            x['split']=np.select([day=='2015-01-22',day=='2015-01-23',day=='2015-02-18'],['train','calibration','test'],default='excluded')
        else:
            x['split']=np.select([day<='2019-04-24',day=='2019-04-25',day.isin(['2019-04-26','2019-04-27'])],['train','calibration','test'],default='excluded')
        x=x[x.split!='excluded'].copy()
        x=purge_groups(x)
        parts={s:x[x.split==s] for s in ['train','calibration','test']}
        for a,b in [('train','calibration'),('train','test'),('calibration','test')]:
            assert not set(parts[a]._group)&set(parts[b]._group)
            assert not set(parts[a]._event_time.dt.floor('D'))&set(parts[b]._event_time.dt.floor('D'))
        for s,z in parts.items():
            STRICT_SPLITS.append(dict(dataset=ds,split=s,n=len(z),benign=int((z.target==0).sum()),attacks=int((z.target==1).sum()),first=str(z._event_time.min()),last=str(z._event_time.max()),purged=json.dumps(x.attrs.get('purged_cross_period',{}))))
        tr,cal,te=[parts[s] for s in ['train','calibration','test']]
        assert tr.target.nunique()==2 and te.target.nunique()==2 and (cal.target==0).sum()>=30
        features=SCHEMAS[ds]['num']+SCHEMAS[ds]['cat']
        x[['_row_key','_group','split']].to_parquet(FINAL/(ds+'_strict_split_rows.parquet'),index=False)
        assert tr._event_time.max()<cal._event_time.min()<te._event_time.min()
        for name,m in models(ds).items():
            if name not in ['LogisticRegression','LightGBM']: continue
            print('STRICT_FIT',ds,name,len(tr),len(cal),len(te),flush=True)
            m.fit(tr[features],tr.target)
            pc=m.predict_proba(cal[features])[:,1]; pt=m.predict_proba(te[features])[:,1]
            CAL_ROWS.extend(calibration_analysis(ds,name,cal,pc,te,pt,FINAL,evaluate,fpr_threshold))
            thr=fpr_threshold(cal.target.to_numpy(),pc)
            assert ((pc[cal.target.to_numpy()==0]>=thr).mean())<=.01000001
            for policy,t in [('default_0.5',.5),('source_cal_fpr_1pct',thr)]:
                r=evaluate(te.target.to_numpy(),pt,t)
                STRICT.append(dict(dataset=ds,protocol='strict_day_disjoint',model=name,policy=policy,threshold=t,calibration_attack_n=int(cal.target.sum()),**r))
                STRICT_PRED.append(pd.DataFrame(dict(row_key=te._row_key.to_numpy(),dataset=ds,model=name,policy=policy,target=te.target.to_numpy(),attack_label=te.attack_label.to_numpy(),score=pt,pred=(pt>=t).astype('int8'),feature_group=te._group.to_numpy(),time_block=te._event_time.dt.floor('h').astype(str).to_numpy())))
            joblib.dump(m,FINAL/(ds+'_'+name+'_strict.joblib'))
            pd.DataFrame(STRICT).to_csv(FINAL/'strict_temporal_metrics.csv',index=False)
            pd.concat(STRICT_PRED).to_parquet(FINAL/'strict_temporal_predictions.parquet',index=False)
    pd.DataFrame(STRICT_SPLITS).to_csv(FINAL/'strict_temporal_split_audit.csv',index=False)
    display(pd.DataFrame(STRICT_SPLITS))
    display(pd.DataFrame(STRICT)[['dataset','model','policy','n','f1','macro_f1','f1_at_20pct_attack','recall','fpr','roc_auc']])
    print('STRICT_COMPLETE',str(FINAL),flush=True)

    pd.DataFrame(CAL_ROWS).to_csv(FINAL/'calibration_sensitivity.csv',index=False)


STRICT_FIT UNSW_NB15 LogisticRegression 73987 2331 101584


STRICT_FIT UNSW_NB15 LightGBM 73987 2331 101584


STRICT_FIT TON_IOT LogisticRegression 39484 78357 61735


STRICT_FIT TON_IOT LightGBM 39484 78357 61735


,dataset,split,n,benign,attacks,first,last,purged
0,UNSW_NB15,train,73987,72455,1532,2015-01-22 11:49:37+00:00,2015-01-22 23:59:59+00:00,"{""calibration"": 4}"
1,UNSW_NB15,calibration,2331,2331,0,2015-01-23 00:00:00+00:00,2015-01-23 00:25:24+00:00,"{""calibration"": 4}"
2,UNSW_NB15,test,101584,80723,20861,2015-02-18 00:23:30+00:00,2015-02-18 12:21:08+00:00,"{""calibration"": 4}"
3,TON_IOT,train,39484,3494,35990,2019-04-02 09:49:44+00:00,2019-04-24 23:59:59+00:00,"{""calibration"": 5, ""test"": 94}"
4,TON_IOT,calibration,78357,1218,77139,2019-04-25 00:00:00+00:00,2019-04-25 23:59:59+00:00,"{""calibration"": 5, ""test"": 94}"
5,TON_IOT,test,61735,1451,60284,2019-04-26 00:00:02+00:00,2019-04-27 01:25:17+00:00,"{""calibration"": 5, ""test"": 94}"


,dataset,model,policy,n,f1,macro_f1,f1_at_20pct_attack,recall,fpr,roc_auc
0,UNSW_NB15,LogisticRegression,default_0.5,101584,0.955075,0.971427,0.953713,0.994631,0.022794,0.998677
1,UNSW_NB15,LogisticRegression,source_cal_fpr_1pct,101584,0.766833,0.840782,0.760854,0.999904,0.157118,0.998677
2,UNSW_NB15,LightGBM,default_0.5,101584,0.961741,0.975791,0.960777,0.982695,0.015733,0.999085
3,UNSW_NB15,LightGBM,source_cal_fpr_1pct,101584,0.707069,0.793584,0.700155,1.000000,0.214127,0.999085
4,TON_IOT,LogisticRegression,default_0.5,61735,0.990208,0.815190,0.636705,0.987177,0.278429,0.909294
5,TON_IOT,LogisticRegression,source_cal_fpr_1pct,61735,0.072094,0.059572,0.068823,0.037406,0.012405,0.909294
6,TON_IOT,LightGBM,default_0.5,61735,0.962429,0.655457,0.690718,0.931856,0.191592,0.865745
7,TON_IOT,LightGBM,source_cal_fpr_1pct,61735,0.000099,0.022721,0.000095,0.000050,0.013094,0.865745


STRICT_COMPLETE run_results


In [7]:
if MODE=='train':
    # Exploratory three-domain evaluation on a deliberately small semantic intersection.
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from lightgbm import LGBMClassifier
    from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score
    import itertools
    CORE=['log_duration','tcp','udp','icmp','other']
    CORE_DATA={}
    for ds,x in DATA.items():
        d=pd.DataFrame(index=x.index)
        duration='dur' if ds=='UNSW_NB15' else 'duration'
        v=pd.to_numeric(x[duration],errors='coerce'); d['log_duration']=np.log1p(v.where(v>=0))
        p=x.proto.astype(str).str.lower().str.strip()
        for k in ['tcp','udp','icmp']: d[k]=(p.str.contains('icmp') if k=='icmp' else p.eq(k)).astype('int8')
        d['other']=(d[['tcp','udp','icmp']].sum(axis=1)==0).astype('int8')
        for c in ['target','attack_label','_row_key','_event_time','source_file']: d[c]=x[c].to_numpy()
        CORE_DATA[ds]=d.reset_index(drop=True)
    r=CLEAN_ROOT
    parts=[]
    for f in sorted((r/'CICIOT2023'/'data').glob('*.parquet')):
        x=pd.read_parquet(f,columns=['flow_duration','TCP','UDP','ICMP','target','attack_label','source_file','source_row'])
        h=pd.util.hash_pandas_object(x[['source_file','source_row']],index=False).to_numpy('uint64')
        x=x.copy() if IS_SAMPLE else x.loc[(h & np.uint64(0xffffffff))/float(2**32)<.025].copy()
        parts.append(x)
    x=pd.concat(parts,ignore_index=True); d=pd.DataFrame(index=x.index)
    v=pd.to_numeric(x.flow_duration,errors='coerce'); d['log_duration']=np.log1p(v.where(v>=0))
    for k in ['tcp','udp','icmp']: d[k]=pd.to_numeric(x[k.upper()],errors='coerce').fillna(0).gt(.5).astype('int8')
    assert (d[['tcp','udp','icmp']].sum(axis=1)<=1).all(), 'CICIoT protocol flags not exclusive'
    d['other']=(d[['tcp','udp','icmp']].sum(axis=1)==0).astype('int8')
    for c in ['target','attack_label','source_file']: d[c]=x[c].to_numpy()
    d['_row_key']='CICIOT2023:'+x.source_file.astype(str)+':'+x.source_row.astype(str)
    d['_event_time']=pd.NaT
    CORE_DATA['CICIOT2023']=d.reset_index(drop=True)
    for ds,d in CORE_DATA.items():
        d['_group']=pd.util.hash_pandas_object(d[CORE],index=False).to_numpy('uint64')
        d['split']=np.where(d._group.to_numpy('uint64')%np.uint64(5)==0,'calibration','train')
        assert d._row_key.is_unique
        print('CORE_SAMPLE',ds,len(d),d.target.mean(),d.groupby('split').target.agg(['size','mean']).to_dict(),flush=True)
    UNSEEN=[]; UNSEEN_PRED=[]; CORE_AUDIT=[]
    for target in CORE_DATA:
        sources=[s for s in CORE_DATA if s!=target]; te=CORE_DATA[target]
        for source_set in [(sources[0],),(sources[1],),tuple(sources)]:
            tr=pd.concat([CORE_DATA[s].query("split=='train'").assign(domain=s) for s in source_set],ignore_index=True)
            cal=pd.concat([CORE_DATA[s].query("split=='calibration'").assign(domain=s) for s in source_set],ignore_index=True)
            # Purge pooled-source feature groups from calibration; target is untouched.
            cal=cal.loc[~cal._group.isin(tr._group)].copy()
            assert not set(tr._group)&set(cal._group)
            assert tr.target.nunique()==2 and all((cal.query('domain==@s').target==0).sum()>=30 for s in source_set)
            weights=np.zeros(len(tr))
            for s in source_set:
                for label in [0,1]:
                    mask=tr.domain.eq(s)&tr.target.eq(label); assert mask.sum()>0
                    weights[mask]=len(tr)/(len(source_set)*2*mask.sum())
            label='+'.join(source_set)
            CORE_AUDIT.append(dict(target=target,source=label,train_n=len(tr),cal_n=len(cal),test_n=len(te),test_core_overlap_with_train=float(te._group.isin(tr._group).mean()),features=';'.join(CORE),status='exploratory exporter semantics and CICIoT duration units not fully validated'))
            for name in ['LogisticRegression','LightGBM']:
                print('UNSEEN_FIT',label,'->',target,name,flush=True)
                est=LogisticRegression(max_iter=500,random_state=42) if name=='LogisticRegression' else LGBMClassifier(n_estimators=220,learning_rate=.05,num_leaves=15,max_depth=6,min_child_samples=80,reg_lambda=8.,verbosity=-1,random_state=42,n_jobs=4)
                model=Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('scale',StandardScaler()),('model',est)])
                model.fit(tr[CORE],tr.target,model__sample_weight=weights)
                pc=model.predict_proba(cal[CORE])[:,1]; pt=model.predict_proba(te[CORE])[:,1]
                thr=max(fpr_threshold(cal.loc[cal.domain.eq(s),'target'],pc[cal.domain.eq(s)]) for s in source_set)
                for policy,t in [('default_0.5',.5),('source_cal_fpr_1pct',thr)]:
                    res=evaluate(te.target.to_numpy(),pt,t)
                    UNSEEN.append(dict(target_domain=target,source_domains=label,model=name,policy=policy,threshold=t,**res))
                    pred=te[['_row_key','_group','source_file','_event_time','target','attack_label']].copy()
                    pred['source_domains']=label; pred['target_domain']=target; pred['model']=name; pred['policy']=policy; pred['score']=pt; pred['pred']=(pt>=t).astype('int8')
                    UNSEEN_PRED.append(pred)
                pd.DataFrame(UNSEEN).to_csv(FINAL/'unseen_domain_metrics.csv',index=False)
                joblib.dump(model,FINAL/(target+'__'+label+'__'+name+'.joblib'))
    pd.DataFrame(CORE_AUDIT).to_csv(FINAL/'core_feature_audit.csv',index=False)
    pd.concat(UNSEEN_PRED,ignore_index=True).to_parquet(FINAL/'unseen_predictions.parquet',index=False)
    display(pd.DataFrame(UNSEEN)[['target_domain','source_domains','model','policy','f1','macro_f1','f1_at_20pct_attack','recall','fpr']])
    print('UNSEEN_COMPLETE',flush=True)


CORE_SAMPLE UNSW_NB15 177906 0.1258698413769069 {'size': {'calibration': 30065, 'train': 147841}, 'mean': {'calibration': 0.058007650091468486, 'train': 0.13967032149403752}}


CORE_SAMPLE TON_IOT 179675 0.9656156950048699 {'size': {'calibration': 29003, 'train': 150672}, 'mean': {'calibration': 0.9779677964348515, 'train': 0.9632380269724965}}


CORE_SAMPLE CICIOT2023 195648 0.9760692672554793 {'size': {'calibration': 17967, 'train': 177681}, 'mean': {'calibration': 0.9481271219457895, 'train': 0.9788947608354298}}


UNSEEN_FIT TON_IOT -> UNSW_NB15 LogisticRegression


UNSEEN_FIT TON_IOT -> UNSW_NB15 LightGBM


UNSEEN_FIT CICIOT2023 -> UNSW_NB15 LogisticRegression


... (13 outputs left out)

UNSEEN_FIT UNSW_NB15+TON_IOT -> CICIOT2023 LogisticRegression


UNSEEN_FIT UNSW_NB15+TON_IOT -> CICIOT2023 LightGBM


,target_domain,source_domains,model,policy,f1,macro_f1,f1_at_20pct_attack,recall,fpr
0,UNSW_NB15,TON_IOT,LogisticRegression,default_0.5,0.067221,0.271578,0.101174,0.192114,0.651392
1,UNSW_NB15,TON_IOT,LogisticRegression,source_cal_fpr_1pct,0.005047,0.468805,0.005060,0.002545,0.000875
2,UNSW_NB15,TON_IOT,LightGBM,default_0.5,0.246075,0.342762,0.356256,0.833698,0.711658
3,UNSW_NB15,TON_IOT,LightGBM,source_cal_fpr_1pct,0.002469,0.467281,0.002481,0.001250,0.001653
4,UNSW_NB15,CICIOT2023,LogisticRegression,default_0.5,0.222612,0.148900,0.331124,0.960881,0.960717
5,UNSW_NB15,CICIOT2023,LogisticRegression,source_cal_fpr_1pct,0.239255,0.341044,0.346835,0.803599,0.707574
6,UNSW_NB15,CICIOT2023,LightGBM,default_0.5,0.229188,0.303247,0.334930,0.813647,0.761242
7,UNSW_NB15,CICIOT2023,LightGBM,source_cal_fpr_1pct,0.248788,0.366677,0.358328,0.804314,0.671236
8,UNSW_NB15,TON_IOT+CICIOT2023,LogisticRegression,default_0.5,0.101304,0.302617,0.150641,0.286250,0.628546
9,UNSW_NB15,TON_IOT+CICIOT2023,LogisticRegression,source_cal_fpr_1pct,0.000000,0.466354,0.000000,0.000000,0.000264


UNSEEN_COMPLETE


## Controlled benchmark — requires training data

Five baselines (constant majority, LR, RF, HGB, LightGBM), three seeds, in-domain, both single-source and pooled-source training. Every comparison uses the same held-out target rows, features, calibration policy and total training budget. Source weights have equal domain/class mass. Protocol-only ablation tests dependence on the unresolved duration proxy. The benchmark is a new exploratory experiment; no values are supplied until execution.

Splits use full-core feature groups independently of labels. Fixed target test subsets allow within-domain minus transfer performance drops on identical rows. Equal count does not equal equal information; learning curves and independent replications remain necessary. Hash-based groups prevent identical recorded vectors crossing partitions, not host/session leakage.

In [8]:
if MODE=='train':
    from itertools import combinations
    BENCH_SEEDS=[42,7,2026]
    B={}
    for domain,d in CORE_DATA.items():
        z=d.copy()
        bucket=z._group.to_numpy('uint64')%np.uint64(10)
        z['partition']=np.where(bucket<6,'train',np.where(bucket<8,'calibration','test'))
        for a,b in combinations(['train','calibration','test'],2):
            assert not set(z.loc[z.partition==a,'_group'])&set(z.loc[z.partition==b,'_group'])
        assert all(z.loc[z.partition==part].target.nunique()==2 for part in ['train','calibration','test'])
        z[['_row_key','_group','partition']].to_parquet(FINAL/(domain+'_benchmark_splits.parquet'),index=False)
        B[domain]=z
    # Same total budget for in-domain, single-source and multi-source; set before target scoring.
    BUDGET=min(60000,min(int((d.partition=='train').sum()) for d in B.values()))
    assert BUDGET>=1000
    def estimator(name,seed):
        return {'Majority':lambda:DummyClassifier(strategy='most_frequent'),
            'LogisticRegression':lambda:LogisticRegression(max_iter=500,random_state=seed),
            'RandomForest':lambda:RandomForestClassifier(n_estimators=110,max_depth=16,min_samples_leaf=5,n_jobs=4,random_state=seed),
            'HistGradientBoosting':lambda:HistGradientBoostingClassifier(max_iter=220,max_leaf_nodes=15,l2_regularization=8,random_state=seed,early_stopping=False),
            'LightGBM':lambda:LGBMClassifier(n_estimators=220,learning_rate=.05,num_leaves=15,max_depth=6,min_child_samples=80,reg_lambda=8,verbosity=-1,n_jobs=4,random_state=seed)}[name]()
    BM=[]; BE=[]; SHIFT=[]; MEMBERS=[]; CONDITIONAL=[]
    for target in B:
        others=[s for s in B if s!=target]
        te=B[target].query("partition=='test'").copy()
        for source_set in [(target,), (others[0],), (others[1],),tuple(others)]:
            source_label='+'.join(source_set)
            scenario='in_domain' if source_set==(target,) else ('multi' if len(source_set)>1 else 'single')
            # Calibration and train use source domains only. Target test is fixed across all scenarios.
            rawtrain=pd.concat([B[s].query("partition=='train'").assign(domain=s) for s in source_set],ignore_index=True)
            cal=pd.concat([B[s].query("partition=='calibration'").assign(domain=s) for s in source_set],ignore_index=True)
            assert not set(rawtrain._group)&set(cal._group)
            assert not set(rawtrain._group)&set(te._group)
            for seed in BENCH_SEEDS:
                chunks=[]
                for j,s in enumerate(source_set):
                    pool=rawtrain[rawtrain.domain.eq(s)]
                    n=BUDGET//len(source_set)+(j<BUDGET%len(source_set))
                    chunks.append(pool.sample(n=n,random_state=seed))
                tr=pd.concat(chunks,ignore_index=True)
                assert len(tr)==BUDGET
                for part,d in [('train',tr),('calibration',cal),('test',te)]:
                    MEMBERS.append(d[['_row_key','_group']].assign(target=target,sources=source_label,seed=seed,partition=part))
                weights=np.zeros(len(tr))
                for s in source_set:
                    for label in [0,1]:
                        mask=tr.domain.eq(s)&tr.target.eq(label); assert mask.sum()>0
                        weights[mask]=len(tr)/(len(source_set)*2*mask.sum())
                # Source-defined numeric bins; KS uses finite values, missingness separately.
                for f in CORE:
                    a=tr[f].replace([np.inf,-np.inf],np.nan).dropna().to_numpy()
                    b=te[f].replace([np.inf,-np.inf],np.nan).dropna().to_numpy()
                    assert len(a) and len(b)
                    cuts=np.unique(np.quantile(a,np.linspace(0,1,11)))
                    edges=np.r_[-np.inf,cuts,np.inf]
                    pa=np.histogram(a,edges)[0]; pb=np.histogram(b,edges)[0]
                    SHIFT.append(dict(target=target,sources=source_label,seed=seed,feature=f,ks=float(ks_2samp(a,b).statistic),jsd_bits=float(jensenshannon(pa/pa.sum(),pb/pb.sum(),base=2)**2),outside_source_range=float(((b<a.min())|(b>a.max())).mean()),source_missing=float(tr[f].isna().mean()),target_missing=float(te[f].isna().mean()),attack_prevalence_delta=float(te.target.mean()-tr.target.mean())))
                    for label in [0,1]:
                        ca=tr.loc[tr.target.eq(label),f].dropna().to_numpy();cb=te.loc[te.target.eq(label),f].dropna().to_numpy()
                        CONDITIONAL.append(dict(target=target,sources=source_label,seed=seed,feature=f,binary_label=label,source_n=len(ca),target_n=len(cb),ks=float(ks_2samp(ca,cb).statistic) if len(ca) and len(cb) else np.nan))
                for variant,features in [('minimal_core',CORE),('protocol_only',['tcp','udp','icmp','other'])]:
                    # Conservative full-core groups also used for ablation. Record collisions after dropping duration.
                    reduced_hash=pd.util.hash_pandas_object(tr[features],index=False)
                    overlap=float(pd.util.hash_pandas_object(te[features],index=False).isin(reduced_hash).mean())
                    external_task = target == 'CICIOT2023' and 'CICIOT2023' not in source_set
                    if external_task:
                        rt_overlap = source_overlap(tr, EXT_X, features)
                    for name in ['Majority','LogisticRegression','RandomForest','HistGradientBoosting','LightGBM']:
                        model=Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('scale',StandardScaler()),('model',estimator(name,seed))])
                        # Majority uses unweighted source frequencies; predictive models use fixed domain/class weighting.
                        kw={} if name=='Majority' else {'model__sample_weight':weights}
                        print('BENCH_FIT',target,source_label,seed,variant,name,flush=True)
                        model.fit(tr[features],tr.target,**kw)
                        # Accumulate tree probabilities in tree order; ties near the quantile are decision-relevant.
                        if name == 'RandomForest':
                            model.named_steps['model'].set_params(n_jobs=1)
                        pc=model.predict_proba(cal[features])[:,1]; pt=model.predict_proba(te[features])[:,1]
                        thr=max(fpr_threshold(cal.loc[cal.domain.eq(s),'target'],pc[cal.domain.eq(s)]) for s in source_set)
                        if external_task:
                            EXT_RUNS.append(score_external(model,features,EXT_X,source_label,seed,variant,name,thr,rt_overlap,FINAL))
                        for policy,t in [('default_0.5',.5),('source_cal_fpr_1pct',thr)]:
                            info=dict(target=target,sources=source_label,scenario=scenario,seed=seed,variant=variant,model=name,policy=policy,threshold=t,train_n=len(tr),test_n=len(te),reduced_feature_overlap=overlap)
                            BM.append(dict(**info,**evaluate(te.target,pt,t)))
                            pred=(pt>=t).astype('int8')
                            g=te[['attack_label','target']].copy();g['pred']=pred
                            for (attack,y),q in g.groupby(['attack_label','target']):
                                BE.append(dict(**info,attack_label=attack,target_label=y,n=len(q),FP=int(((q.target==0)&(q.pred==1)).sum()),FN=int(((q.target==1)&(q.pred==0)).sum())))
                        path=FINAL/'benchmark_predictions';path.mkdir(exist_ok=True)
                        te[['_row_key','_group','target','attack_label']].assign(score=pt,threshold=thr).to_parquet(path/(target+'__'+source_label+'__'+str(seed)+'__'+variant+'__'+name+'.parquet'),index=False)
                pd.DataFrame(BM).to_csv(FINAL/'controlled_benchmark.csv',index=False)
    bm=pd.DataFrame(BM)
    ref=bm[bm.scenario.eq('in_domain')][['target','seed','variant','model','policy','macro_f1','fpr']].rename(columns={'macro_f1':'in_domain_macro_f1','fpr':'in_domain_fpr'})
    drops=bm.merge(ref,on=['target','seed','variant','model','policy'],validate='many_to_one')
    drops['macro_f1_drop']=drops.in_domain_macro_f1-drops.macro_f1
    drops['fpr_increase']=drops.fpr-drops.in_domain_fpr
    drops.to_csv(FINAL/'benchmark_performance_drop.csv',index=False)
    pd.DataFrame(BE).to_csv(FINAL/'benchmark_attack_errors.csv',index=False)
    shift=pd.DataFrame(SHIFT);shift.to_csv(FINAL/'benchmark_feature_shift.csv',index=False)
    pd.DataFrame(CONDITIONAL).to_csv(FINAL/'benchmark_class_conditional_shift.csv',index=False)
    pd.concat(MEMBERS,ignore_index=True).to_parquet(FINAL/'benchmark_membership.parquet',index=False)
    bm.groupby(['target','sources','variant','model','policy'])[['macro_f1','recall','fpr','f1_at_20pct_attack']].agg(['mean','std','min','max']).to_csv(FINAL/'benchmark_seed_summary.csv')
    # Joined conditions for descriptive plots only; repeated models/seeds are not independent shift environments.
    joined=drops.merge(shift.groupby(['target','sources','seed']).ks.mean().rename('mean_ks'),on=['target','sources','seed'],validate='many_to_one')
    joined.to_csv(FINAL/'shift_performance_association.csv',index=False)
    (FINAL/'benchmark_protocol.json').write_text(json.dumps(dict(seeds=BENCH_SEEDS,total_training_budget=BUDGET,features=CORE,target_test='fixed hash groups 8 and 9',threshold='max source calibration benign 99th percentile nextafter',status='executed'),indent=2))
else:
    print('PENDING: controlled four-model benchmark, majority baseline, equal-budget multi-source, seed sensitivity, feature-shift and duration ablation require Kaggle cleaned data.')


BENCH_FIT UNSW_NB15 UNSW_NB15 42 minimal_core Majority


BENCH_FIT UNSW_NB15 UNSW_NB15 42 minimal_core LogisticRegression


BENCH_FIT UNSW_NB15 UNSW_NB15 42 minimal_core RandomForest


BENCH_FIT UNSW_NB15 UNSW_NB15 42 minimal_core HistGradientBoosting


BENCH_FIT UNSW_NB15 UNSW_NB15 42 minimal_core LightGBM


BENCH_FIT UNSW_NB15 UNSW_NB15 42 protocol_only Majority


... (351 outputs left out)

BENCH_FIT CICIOT2023 UNSW_NB15+TON_IOT 2026 protocol_only LogisticRegression


BENCH_FIT CICIOT2023 UNSW_NB15+TON_IOT 2026 protocol_only RandomForest


BENCH_FIT CICIOT2023 UNSW_NB15+TON_IOT 2026 protocol_only HistGradientBoosting


BENCH_FIT CICIOT2023 UNSW_NB15+TON_IOT 2026 protocol_only LightGBM


## Native-feature artifact ablation

Identical day splits, rows and thresholds; train-only transformations; five baselines and three seeds. Conservative features are explicitly listed before scoring and remove TTL, ports, TCP sequence/window/timing fields, contextual counters and application response/status fields. This is a sensitivity analysis of offline flow classification; completed-flow fields are not claimed available for early online detection. Error strata and within-label shift are descriptive.

In [9]:
if MODE=='train':
    import copy
    from sklearn.base import clone
    CONSERVATIVE={
        'UNSW_NB15':{'num':['dur','sbytes','dbytes','Spkts','Dpkts','smeansz','dmeansz'],'cat':['proto']},
        'TON_IOT':{'num':['duration','src_bytes','dst_bytes','src_pkts','dst_pkts','src_ip_bytes','dst_ip_bytes'],'cat':['proto']}}
    NR=[];NE=[];NS=[];NM=[]
    for ds,raw in DATA.items():
        x=raw.dropna(subset=['_event_time']).copy();day=x._event_time.dt.strftime('%Y-%m-%d')
        if ds=='UNSW_NB15':x['split']=np.select([day=='2015-01-22',day=='2015-01-23',day=='2015-02-18'],['train','calibration','test'],default='excluded')
        else:x['split']=np.select([day<='2019-04-24',day=='2019-04-25',day.isin(['2019-04-26','2019-04-27'])],['train','calibration','test'],default='excluded')
        x=purge_groups(x[x.split!='excluded'].copy())
        tr,cal,te=[x[x.split.eq(s)].copy() for s in ['train','calibration','test']]
        native=copy.deepcopy(SCHEMAS[ds])
        conservative={k:[f for f in CONSERVATIVE[ds][k] if f in native[k]] for k in ['num','cat']}
        assert conservative['num'] and conservative['cat']
        for variant,cfg in [('native',native),('conservative',conservative)]:
            SCHEMAS[ds]=cfg
            prep=build_preprocessor(ds)
            features=cfg['num']+cfg['cat']
            NM.append(dict(dataset=ds,variant=variant,features=features,removed=[f for f in native['num']+native['cat'] if f not in features]))
            for f in cfg['num']:
                for target_label in ['all',0,1]:
                    a=tr[f] if target_label=='all' else tr.loc[tr.target.eq(target_label),f]
                    b=te[f] if target_label=='all' else te.loc[te.target.eq(target_label),f]
                    a=a.dropna().to_numpy();b=b.dropna().to_numpy()
                    NS.append(dict(dataset=ds,variant=variant,feature=f,binary_label=str(target_label),train_n=len(a),test_n=len(b),ks=float(ks_2samp(a,b).statistic) if len(a) and len(b) else np.nan))
            for seed in [42,7,2026]:
                for name in ['Majority','LogisticRegression','RandomForest','HistGradientBoosting','LightGBM']:
                    model=Pipeline([('preprocess',clone(prep)),('model',estimator(name,seed))])
                    print('NATIVE_FIT',ds,variant,seed,name,flush=True)
                    model.fit(tr[features],tr.target)
                    # Accumulate tree probabilities in tree order; ties near the quantile are decision-relevant.
                    if name == 'RandomForest':
                        model.named_steps['model'].set_params(n_jobs=1)
                    pc=model.predict_proba(cal[features])[:,1];pt=model.predict_proba(te[features])[:,1]
                    threshold=fpr_threshold(cal.target,pc)
                    for policy,t in [('default_0.5',.5),('source_cal_fpr_1pct',threshold)]:
                        info=dict(dataset=ds,variant=variant,seed=seed,model=name,policy=policy,threshold=t,train_n=len(tr))
                        NR.append(dict(**info,**evaluate(te.target,pt,t)))
                        q=te[['attack_label','target']].copy();q['pred']=(pt>=t).astype('int8')
                        q['protocol']=te.proto.astype(str).to_numpy()
                        for (attack,y,proto),g in q.groupby(['attack_label','target','protocol']):
                            NE.append(dict(**info,attack_label=attack,target_label=y,protocol=proto,n=len(g),FP=int(((g.target==0)&g.pred.eq(1)).sum()),FN=int(((g.target==1)&g.pred.eq(0)).sum())))
                    folder=FINAL/'native_predictions';folder.mkdir(exist_ok=True)
                    te[['_row_key','_group','target','attack_label','_event_time']].assign(score=pt,threshold=threshold).to_parquet(folder/(ds+'__'+variant+'__'+str(seed)+'__'+name+'.parquet'),index=False)
                    pd.DataFrame(NR).to_csv(FINAL/'native_ablation.csv',index=False)
            SCHEMAS[ds]=native
        # Composition shifts within each dataset use original labels, not harmonized cross-domain attack taxonomies.
        mix=pd.crosstab(x.attack_label,x.split).reindex(columns=['train','calibration','test'],fill_value=0)
        mix.to_csv(FINAL/(ds+'_temporal_attack_composition.csv'))
        for a,b in [('train','calibration'),('train','test')]:
            value=float(jensenshannon(mix[a]/mix[a].sum(),mix[b]/mix[b].sum(),base=2)**2)
            NS.append(dict(dataset=ds,variant='attack_mix',feature='attack_label',binary_label='all',comparison=a+'->'+b,jsd_bits=value))
    pd.DataFrame(NE).to_csv(FINAL/'native_attack_protocol_errors.csv',index=False)
    pd.DataFrame(NS).to_csv(FINAL/'native_feature_shift.csv',index=False)
    (FINAL/'native_feature_manifest.json').write_text(json.dumps(NM,indent=2))
    nr=pd.DataFrame(NR)
    ref=nr[nr.variant.eq('native')][['dataset','seed','model','policy','macro_f1','fpr']].rename(columns={'macro_f1':'native_macro_f1','fpr':'native_fpr'})
    sensitivity=nr.merge(ref,on=['dataset','seed','model','policy'],validate='many_to_one')
    sensitivity['delta_macro_f1']=sensitivity.macro_f1-sensitivity.native_macro_f1
    sensitivity['delta_fpr']=sensitivity.fpr-sensitivity.native_fpr
    sensitivity.to_csv(FINAL/'native_ablation_paired_deltas.csv',index=False)
    nr.groupby(['dataset','variant','model','policy'])[['macro_f1','recall','fpr','f1_at_20pct_attack']].agg(['mean','std','min','max']).to_csv(FINAL/'native_seed_summary.csv')


NATIVE_FIT UNSW_NB15 native 42 Majority


NATIVE_FIT UNSW_NB15 native 42 LogisticRegression


NATIVE_FIT UNSW_NB15 native 42 RandomForest


NATIVE_FIT UNSW_NB15 native 42 HistGradientBoosting


NATIVE_FIT UNSW_NB15 native 42 LightGBM


NATIVE_FIT UNSW_NB15 native 7 Majority


... (50 outputs left out)

NATIVE_FIT TON_IOT conservative 2026 LogisticRegression


NATIVE_FIT TON_IOT conservative 2026 RandomForest


NATIVE_FIT TON_IOT conservative 2026 HistGradientBoosting


NATIVE_FIT TON_IOT conservative 2026 LightGBM


## Внешняя проверка на RT-IoT2022

Метки открываются для расчета метрик после обучения всей матрицы. На новом наборе используются сохраненные оценки тех же экземпляров моделей, что предсказывали CICIoT2023. Обучающие источники — UNSW-NB15, ToN-IoT и их объединение.

In [10]:
if MODE == 'train':
    evaluate_external(EXT_Y, EXT_RUNS, FINAL, evaluate)
    print('External evaluation:', len(EXT_RUNS), 'source fits,', len(EXT_Y), 'test records')

External evaluation: 90 source fits, 123117 test records


## Saved-result verification

Replay checks correspondence of predictions and all central exported metrics; it cannot establish train/test separation without raw training partitions. The stored split audit is supporting evidence, not an independent raw-data leakage check. Historical uncertainty is recomputed conditionally on fixed models. UTC-hour and feature-vector clusters are imperfect dependence units; cluster-swap p-values are exploratory and require exchangeability. Holm covers the 28 historical primary comparisons only. New benchmark seeds are reported descriptively, not pooled as independent deployments.

In [11]:
if MODE=='replay':
    required=['strict_temporal_predictions.parquet','unseen_predictions.parquet','strict_temporal_metrics.csv','unseen_domain_metrics.csv']
    assert all((SAVED/f).exists() for f in required), 'Place saved_results beside notebook or set NIDS_SAVED.'
    SP=pd.read_parquet(SAVED/required[0]).rename(columns={'row_key':'_row_key','feature_group':'_group'})
    UP=pd.read_parquet(SAVED/required[1])
    STRICT_PRED=[SP]; UNSEEN_PRED=[UP]
    STRICT=pd.read_csv(SAVED/required[2],float_precision='round_trip').to_dict('records')
    UNSEEN=pd.read_csv(SAVED/required[3],float_precision='round_trip').to_dict('records')
else:
    SP=pd.concat(STRICT_PRED,ignore_index=True).rename(columns={'row_key':'_row_key','feature_group':'_group'})
    UP=pd.concat(UNSEEN_PRED,ignore_index=True)
checks=[]; dummy=[]; errors=[]
for typ,preds,records,keys in [('strict',SP,STRICT,['dataset','model','policy']),('unseen',UP,UNSEEN,['target_domain','source_domains','model','policy'])]:
    table=pd.DataFrame(records)
    for key,z in preds.groupby(keys,sort=True):
        assert z._row_key.is_unique and z.target.isin([0,1]).all() and np.isfinite(z.score).all()
        mask=np.ones(len(table),dtype=bool)
        for k,v in zip(keys,key): mask &= table[k].eq(v).to_numpy()
        assert mask.sum()==1
        old=table.loc[mask].iloc[0]
        assert np.array_equal(z.pred.to_numpy(),(z.score>=old.threshold).astype('int8').to_numpy()), (key,'threshold mismatch')
        calc=evaluate(z.target,z.score,old.threshold)
        for metric in ['n','tn','fp','fn','tp','accuracy','precision','recall','f1','macro_f1','fpr','roc_auc','average_precision','f1_at_20pct_attack']:
            assert np.isclose(calc[metric],old[metric],atol=1e-10,equal_nan=True), (key,metric,calc[metric],old[metric])
        checks.append(dict(experiment=typ,key=str(key),n=len(z),status='PASS',checked='row identity, threshold, confusion counts, 14 central metrics'))
        for label,prediction in [('always_benign',0),('always_attack',1)]:
            # Constant-rule references, not fitted majority baselines or target-selected models.
            val=evaluate(z.target,np.full(len(z),prediction,dtype=float),.5)
            dummy.append(dict(experiment=typ,**dict(zip(keys,key)),rule=label,**val))
        for (attack,y),g in z.groupby(['attack_label','target']):
            errors.append(dict(experiment=typ,**dict(zip(keys,key)),attack_label=attack,target=y,n=len(g),FP=int(((g.target==0)&(g.pred==1)).sum()),FN=int(((g.target==1)&(g.pred==0)).sum()),error_rate=float((g.target!=g.pred).mean())))
pd.DataFrame(checks).to_csv(FINAL/'metric_verification.csv',index=False)
pd.DataFrame(dummy).to_csv(FINAL/'constant_rule_comparisons.csv',index=False)
pd.DataFrame(errors).to_csv(FINAL/'fp_fn_by_attack.csv',index=False)
print('Verified prediction groups:',len(checks))


Verified prediction groups: 44


In [12]:
# Conditional uncertainty, paired comparisons, and reproducibility export.
import scipy, sklearn, lightgbm, shutil, hashlib
from scipy.stats import binomtest
from IPython.display import FileLink
REPS=1000
rng=np.random.default_rng(42)
def from_counts(c):
    c=np.asarray(c,dtype=float); tn,fp,fn,tp=np.moveaxis(c,-1,0)
    with np.errstate(invalid='ignore',divide='ignore'):
        f1=2*tp/(2*tp+fp+fn); fb=2*tn/(2*tn+fp+fn)
        rec=tp/(tp+fn); fpr=fp/(fp+tn)
        return np.stack([(f1+fb)/2,2*.2*rec/(.2*(1+rec)+.8*fpr),rec,fpr],axis=-1)
def groups_for(z,temporal):
    return z.time_block.astype(str) if temporal else z._group.astype(str)
def counts_by_cluster(z,temporal):
    k=groups_for(z,temporal)
    # Order TN, FP, FN, TP.
    a=pd.DataFrame({'cluster':k.to_numpy(),'tn':((z.target==0)&(z.pred==0)).astype(int).to_numpy(),'fp':((z.target==0)&(z.pred==1)).astype(int).to_numpy(),'fn':((z.target==1)&(z.pred==0)).astype(int).to_numpy(),'tp':((z.target==1)&(z.pred==1)).astype(int).to_numpy()})
    return a.groupby('cluster')[['tn','fp','fn','tp']].sum()
def boot_pair(a,b,temporal):
    a=a.sort_values('_row_key').reset_index(drop=True); b=b.sort_values('_row_key').reset_index(drop=True)
    assert a._row_key.equals(b._row_key) and np.array_equal(a.target,b.target)
    ca=counts_by_cluster(a,temporal); cb=counts_by_cluster(b,temporal).reindex(ca.index)
    assert not cb.isna().any().any() and groups_for(a,temporal).equals(groups_for(b,temporal))
    aa=ca.to_numpy(); bb=cb.to_numpy(); n=len(aa)
    assert n>=2, 'At least two clusters required'
    pair,freq=np.unique(np.column_stack([aa,bb]),axis=0,return_counts=True); aa=pair[:,:4];bb=pair[:,4:]
    observed=from_counts(freq@aa)-from_counts(freq@bb); vals=[]; null=[]
    for _ in range(REPS):
        w=rng.multinomial(n,freq/n); vals.append(from_counts(w@aa)-from_counts(w@bb))
        swapped=rng.binomial(freq,.5)
        na=(freq-swapped)@aa+swapped@bb; nb=(freq-swapped)@bb+swapped@aa
        null.append(from_counts(na)[0]-from_counts(nb)[0])
    vals=np.asarray(vals); lo,hi=np.nanquantile(vals,[.025,.975],axis=0)
    null=np.asarray(null); null=null[np.isfinite(null)]
    assert len(null)>0 and np.isfinite(observed[0])
    p=(1+np.sum(np.abs(null)>=abs(observed[0])))/(len(null)+1)
    out=dict(clusters=n,resamples=REPS,valid_null_resamples=len(null),p_macro_f1_randomization=p,valid_macro_bootstrap=int(np.isfinite(vals[:,0]).sum()),cluster_basis='UTC hour' if temporal else 'canonical feature vector')
    for j,m in enumerate(['macro_f1','f1_at_20pct_attack','recall','fpr']): out.update({m+'_delta':observed[j],m+'_delta_ci_low':lo[j],m+'_delta_ci_high':hi[j]})
    return out
SP=pd.concat(STRICT_PRED,ignore_index=True).rename(columns={'row_key':'_row_key','feature_group':'_group'})
UP=pd.concat(UNSEEN_PRED,ignore_index=True)
COMPS=[]; CI=[]; ERR=[]
for typ,preds,keys in [('strict',SP,['dataset','model','policy']),('unseen',UP,['target_domain','source_domains','model','policy'])]:
    for key,z in preds.groupby(keys,sort=False):
        info=dict(zip(keys,key)); temporal=typ=='strict'
        c=counts_by_cluster(z,temporal).to_numpy(); ng=len(c); vals=[]
        c,freq=np.unique(c,axis=0,return_counts=True)
        for _ in range(REPS):
            w=rng.multinomial(ng,freq/ng); vals.append(from_counts(w@c))
        vals=np.asarray(vals); lo,hi=np.nanquantile(vals,[.025,.975],axis=0)
        rec=dict(experiment=typ,**info,clusters=ng,resamples=REPS,cluster_basis='UTC hour' if temporal else 'canonical feature vector')
        for j,m in enumerate(['macro_f1','f1_at_20pct_attack','recall','fpr']): rec.update({m+'_ci_low':lo[j],m+'_ci_high':hi[j]})
        rec['valid_macro_bootstrap']=int(np.isfinite(vals[:,0]).sum())
        CI.append(rec)
        for (attack,target_value),g in z.groupby(['attack_label','target']):
            ERR.append(dict(experiment=typ,**info,attack_label=attack,n=len(g),target=int(g.target.iloc[0]),predicted_attack=int(g.pred.sum()),error_n=int((g.pred!=g.target).sum())))
for (ds,policy),z in SP.groupby(['dataset','policy']):
    a=z[z.model=='LightGBM'];b=z[z.model=='LogisticRegression']
    COMPS.append(dict(experiment='strict',target_domain=ds,policy=policy,model='LightGBM vs LogisticRegression',baseline='LogisticRegression',**boot_pair(a,b,True)))
for (target,name,policy),z in UP.groupby(['target_domain','model','policy']):
    sources=z.source_domains.unique(); multi=[s for s in sources if '+' in s][0]
    for single in [s for s in sources if '+' not in s]:
        COMPS.append(dict(experiment='unseen',target_domain=target,model=name,policy=policy,baseline=single,**boot_pair(z[z.source_domains==multi],z[z.source_domains==single],False)))
comparisons=pd.DataFrame(COMPS); p=comparisons.p_macro_f1_randomization.to_numpy(); order=np.argsort(p); adj=np.maximum.accumulate(p[order]*(len(p)-np.arange(len(p)))); adjusted=np.empty(len(p));adjusted[order]=np.minimum(adj,1)
comparisons['p_macro_f1_holm']=adjusted
comparisons.to_csv(FINAL/'paired_statistical_comparisons.csv',index=False)
pd.DataFrame(CI).to_csv(FINAL/'conditional_cluster_confidence_intervals.csv',index=False)
pd.DataFrame(ERR).to_csv(FINAL/'attack_error_analysis.csv',index=False)


In [13]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
F=FINAL/'figures';F.mkdir(exist_ok=True)
st=pd.DataFrame(STRICT); un=pd.DataFrame(UNSEEN)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,metric in zip(axes,['recall','fpr']):
    st.assign(label=st.dataset+' | '+st.model).pivot(index='label',columns='policy',values=metric).plot.barh(ax=ax)
    ax.set_xlim(0,1);ax.set_xlabel(metric);ax.set_ylabel('')
fig.suptitle('Strict day-disjoint test: saved predictions');fig.tight_layout();fig.savefig(F/'temporal_operating_points.png',dpi=160);plt.close(fig)
for policy in un.policy.unique():
    z=un[un.policy.eq(policy)].copy();z['setting']=z.target_domain+' <- '+z.source_domains
    fig,ax=plt.subplots(figsize=(9,6));z.pivot(index='setting',columns='model',values='macro_f1').plot.barh(ax=ax)
    ax.set_xlim(0,1);ax.set_xlabel('Macro-F1');ax.set_ylabel('');ax.set_title(policy);fig.tight_layout();fig.savefig(F/('transfer_'+policy+'.png'),dpi=160);plt.close(fig)
fig,axes=plt.subplots(2,2,figsize=(10,8))
for row,(ds,z) in enumerate(SP[SP.policy.eq('default_0.5')].groupby('dataset')):
    for name,g in z.groupby('model'):
        RocCurveDisplay.from_predictions(g.target,g.score,name=name,ax=axes[row,0])
        PrecisionRecallDisplay.from_predictions(g.target,g.score,name=name,ax=axes[row,1])
    axes[row,0].set_title(ds+' ROC');axes[row,1].set_title(ds+' precision-recall')
fig.tight_layout();fig.savefig(F/'temporal_roc_pr.png',dpi=160);plt.close(fig)
err=pd.DataFrame(errors)
z=err[(err.experiment=='strict')&(err.policy=='default_0.5')].copy()
z['label']=z.dataset+' | '+z.model+' | '+z.attack_label.astype(str)+' [n='+z.n.astype(str)+']'
fig,ax=plt.subplots(figsize=(10,max(5,len(z)*.22)));z.set_index('label').error_rate.plot.barh(ax=ax)
ax.set_xlim(0,1);ax.set_xlabel('Error rate; benign = false positives, attack = false negatives');fig.tight_layout();fig.savefig(F/'attack_error_rates.png',dpi=140);plt.close(fig)
ci=pd.DataFrame(CI);fig,ax=plt.subplots(figsize=(9,4))
z=ci[ci.experiment.eq('strict')].reset_index(drop=True)
center=st.merge(z,on=['dataset','model','policy'],validate='one_to_one')
labels=center.dataset+' | '+center.model+' | '+center.policy
ax.errorbar(center.macro_f1,range(len(center)),xerr=np.maximum(0,np.vstack([center.macro_f1-center.macro_f1_ci_low,center.macro_f1_ci_high-center.macro_f1])),fmt='o')
ax.set_yticks(range(len(center)),labels);ax.set_xlim(0,1);ax.set_xlabel('Macro-F1: conditional 95% cluster bootstrap interval');fig.tight_layout();fig.savefig(F/'conditional_intervals.png',dpi=160);plt.close(fig)
# Operational temporal error profile from saved hour blocks, no invented feature measurements.
q=SP[SP.policy.eq('default_0.5')].copy();q['FP']=((q.target==0)&(q.pred==1)).astype(int);q['FN']=((q.target==1)&(q.pred==0)).astype(int)
q['benign']=(q.target==0).astype(int);q['attack']=(q.target==1).astype(int)
hour=q.groupby(['dataset','model','time_block'])[['FP','FN','benign','attack']].sum().reset_index()
hour['fpr']=hour.FP/hour.benign.replace(0,np.nan);hour['fnr']=hour.FN/hour.attack.replace(0,np.nan)
hour.to_csv(FINAL/'hourly_errors.csv',index=False)
fig,axes=plt.subplots(2,1,figsize=(10,6))
for ax,(ds,g) in zip(axes,hour.groupby('dataset')):
    for name,v in g.groupby('model'):ax.plot(pd.to_datetime(v.time_block),v.fpr,marker='.',label=name)
    ax.set_title(ds);ax.set_ylabel('FPR per UTC hour');ax.set_ylim(0,1);ax.legend()
fig.autofmt_xdate();fig.tight_layout();fig.savefig(F/'hourly_fpr.png',dpi=160);plt.close(fig)
if MODE=='train':
    z=joined[(joined.policy=='default_0.5')&(joined.variant=='minimal_core')&(joined.scenario!='in_domain')]
    fig,ax=plt.subplots(figsize=(7,5))
    for name,g in z.groupby('model'):ax.scatter(g.mean_ks,g.macro_f1_drop,label=name,alpha=.6)
    ax.set_xlabel('Mean source-train to target-test KS');ax.set_ylabel('In-domain minus transfer macro-F1');ax.legend();fig.tight_layout();fig.savefig(F/'shift_vs_drop_descriptive.png',dpi=160);plt.close(fig)
print('Figures:',len(list(F.glob('*.png'))))


Figures: 8


In [14]:
status={'mode':MODE,'saved_prediction_groups_verified':len(checks),'historical_comparisons_recomputed':len(COMPS),'training_rerun':MODE=='train','controlled_benchmark': 'completed' if MODE=='train' else 'PENDING_KAGGLE_DATA','partition_assertions_executed': MODE=='train','all_semantic_leakage_excluded':False,'previous_v2_targets_preinspected':True,'external_test_after_local_freeze':MODE=='train','independent_team_replication':False}
(FINAL/'execution_status.json').write_text(json.dumps(status,indent=2))
files=[]
for f in sorted(FINAL.rglob('*')):
    if f.is_file() and f.name!='output_sha256.csv': files.append(dict(file=str(f.relative_to(FINAL)),sha256=hashlib.sha256(f.read_bytes()).hexdigest()))
pd.DataFrame(files).to_csv(FINAL/'output_sha256.csv',index=False)
display(status)


{'mode': 'train',
 'saved_prediction_groups_verified': 44,
 'historical_comparisons_recomputed': 28,
 'training_rerun': True,
 'controlled_benchmark': 'completed',
 'partition_assertions_executed': True,
 'all_semantic_leakage_excluded': False,
 'previous_v2_targets_preinspected': True,
 'external_test_after_local_freeze': True,
 'independent_team_replication': False}

## Границы вывода

Три seed используют один и тот же тест. RT-IoT2022 добавляет один внешний лабораторный набор. Проект не оценивает межорганизационную дисперсию по независимой выборке сетей. Внутридневные окна калибровки относятся к уже изученным дням.